## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-07-55-37 +0000,nypost,Career criminal captured in only his underwear...,https://nypost.com/2026/11/04/us-news/colorado...
1,2026-11-04-07-52-38 +0000,nyt,How Gina Hinojosa Captured Texas,https://www.nytimes.com/2026/11/04/us/gina-hin...
2,2026-11-04-07-52-06 +0000,nyt,Idaho Voters Reject Referendum to Lift State’s...,https://www.nytimes.com/2026/11/04/us/idaho-ab...
3,2026-11-04-07-50-00 +0000,wsj,Voters delivered an astonishing rebuke to Pres...,https://www.wsj.com/politics/elections/five-ta...
4,2026-11-04-07-48-47 +0000,nyt,El-Sayed Wins Michigan Senate Race in a Victor...,https://www.nytimes.com/2026/11/04/us/michigan...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2918/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
40,race,51
24,trump,48
42,democrats,45
353,election,41
39,senate,39


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
151,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...,268
275,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,261
207,2026-11-04-02-18-11 +0000,chitri,Texas Senate race: Democrat James Talarico bea...,https://www.chicagotribune.com/2026/11/03/texa...,240
100,2026-11-04-05-28-25 +0000,chitri,"Iowa governor race: Rob Sand wins election, fl...",https://www.chicagotribune.com/2026/11/03/rob-...,239
196,2026-11-04-03-14-56 +0000,chitri,South Carolina Senate race: Democrat Annie And...,https://www.chicagotribune.com/2026/11/03/sout...,238


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
151,268,2026-11-04-04-40-50 +0000,chitri,Iowa Senate race: Democrat Josh Turek wins ele...,https://www.chicagotribune.com/2026/11/03/josh...
275,261,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
157,104,2026-11-04-04-33-12 +0000,chitri,Lake County Sheriff John Idleburg declares vic...,https://www.chicagotribune.com/2026/11/03/lake...
147,103,2026-11-04-04-43-53 +0000,chitri,Wisconsin governor race: Democrat David Crowle...,https://www.chicagotribune.com/2026/11/03/demo...
231,97,2026-11-04-01-00-20 +0000,chitri,Voters usher in new era with state Rep. La Sha...,https://www.chicagotribune.com/2026/11/03/illi...
22,90,2026-11-04-07-29-00 +0000,wsj,A political reckoning has shattered Trump’s gr...,https://www.wsj.com/politics/elections/a-polit...
299,72,2026-11-03-22-01-05 +0000,nyt,How Will the Midterms Play Out? These Races Ma...,https://www.nytimes.com/2026/11/03/us/politics...
130,58,2026-11-04-05-07-00 +0000,wsj,President Trump said on Tuesday night that the...,https://www.wsj.com/politics/elections/trump-a...
225,56,2026-11-04-01-04-38 +0000,chitri,Georgia Senate race: Sen. Jon Ossoff wins reel...,https://www.chicagotribune.com/2026/11/03/geor...
473,54,2026-11-03-11-05-43 +0000,chitri,‘Absolutely astonishing’: Pastry fans line up ...,https://www.chicagotribune.com/2026/11/03/trit...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
